In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from datetime import datetime as dt
import numpy as np

In [ ]:
colors = [
    'tab:blue',
    'tab:orange',
    'tab:green',
    'tab:red',
    'tab:purple',
    'tab:brown',
    'tab:pink',
    'tab:gray',
    'tab:olive',
    'tab:cyan',
    'teal',
    'peru',
    'fuchsia',
    'yellow',
    'lime',
    'darkviolet'
]

In [ ]:
# Read CPU-sensor history from archive_sort_cleanup
cpus = pd.read_csv('cpus.csv', index_col=0)
dates = [dt.strptime(d, '%Y%m%d') for d in cpus.columns]
all_sensor_nums = np.unique(np.sort(cpus.values.flatten()))

In [ ]:
# Plot CPU-sensor history
px = 1/plt.rcParams['figure.dpi']  # pixel in inches
fig = plt.figure(figsize=(1920*px, 1080*px))
ax = fig.add_subplot(111)
for i, (this_serial, row) in enumerate(cpus.iterrows()):
    row_to_plot = row.replace(0, float('nan'))
    color = colors[i]
    ax.scatter(dates, row_to_plot, label=this_serial, color=color)
    median_date = np.median(np.array(dates)[~np.isnan(row_to_plot)].astype('datetime64[D]').astype(float)).astype('datetime64[D]')
    median_sensor = np.median(row_to_plot[~np.isnan(row_to_plot)])
    ax.text(median_date, median_sensor, this_serial, fontsize=12, ha='center', va='center', color='k')
ax.set_xlabel('Date')
ax.set_ylabel('Sensor Number')
ax.legend()
ax.grid()

In [ ]:
# Create sensor history dataframe
sa_metadata = pd.DataFrame(columns=['start_dt','end_dt','sensor_num','cpu_serial'])

In [ ]:
# Associate each sensor with the date ranges during which it was active on each CPU
for sensor_num in all_sensor_nums:
    if sensor_num == 0:
        continue
    cpus_with_sensor = cpus.where(cpus == sensor_num).dropna(how='all').dropna(axis=1, how='all')
    for cpu_id, this_cpu_data in cpus_with_sensor.iterrows():
        this_date_data = (~this_cpu_data.isna())
        date_changes = np.gradient(this_date_data.astype(float))
        end_dates = this_cpu_data.index[(date_changes == -0.5) & (this_date_data)]
        start_dates = this_cpu_data.index[(date_changes == 0.5) & (this_date_data)]
        if this_date_data.iloc[0]:
            start_dates = np.insert(start_dates, 0, this_cpu_data.index[0])
        if this_date_data.iloc[-1]:
            end_dates = np.append(end_dates, this_cpu_data.index[-1])
        if this_date_data.sum() == 1:
            start_dates = np.array([this_cpu_data.index[this_date_data][0]])
            end_dates = np.array([this_cpu_data.index[this_date_data][0]])
        start_dates = [dt.strptime(sd, '%Y%m%d') for sd in start_dates]
        end_dates = [dt.strptime(ed, '%Y%m%d').replace(hour=23, minute=59, second=59) for ed in end_dates]
        print(f'Sensor {sensor_num} was active on CPU {cpu_id} during the following date ranges:')
        for start, end in zip(start_dates, end_dates):
            print(f'  {start} to {end}')
            sa_metadata_for_sensor = {'start_dt' : start, 'end_dt' : end, 'sensor_num' : sensor_num, 'cpu_serial' : cpu_id}
            sa_metadata_for_sensor = pd.DataFrame(sa_metadata_for_sensor, index=[0])
            sa_metadata = pd.concat([sa_metadata, sa_metadata_for_sensor], ignore_index=True)

In [ ]:
# Affix start date for initial batch of sensors
start_date = dt(2023, 1, 1, 0, 0, 0)
for sensor_num in all_sensor_nums:
    if sensor_num == 0:
        continue
    sa_metadata_for_sensor = sa_metadata[sa_metadata['sensor_num'] == sensor_num]
    first_entry = np.argmin(sa_metadata_for_sensor['start_dt'])
    sa_metadata.loc[sa_metadata_for_sensor.index[first_entry], 'start_dt'] = start_date

In [ ]:
sa_metadata = sa_metadata.sort_values(by=['start_dt', 'sensor_num']).reset_index(drop=True)
for sensor_num in all_sensor_nums:
    if sensor_num == 0:
        continue
    sa_metadata_for_sensor = sa_metadata[sa_metadata['sensor_num'] == sensor_num]
    start_dts = sa_metadata_for_sensor['start_dt'].values
    end_dts = sa_metadata_for_sensor['end_dt'].values
    if len(start_dts) > 1:
        for i in range(len(start_dts)-1):
            if start_dts[i+1] < end_dts[i]:
                print(f'Warning: Sensor {sensor_num} has overlapping date ranges between\n{start_dts[i].strftime("%Y-%m-%d")} through {end_dts[i].strftime("%Y-%m-%d")} and\n{start_dts[i+1].strftime("%Y-%m-%d")} through {end_dts[i+1].strftime("%Y-%m-%d")}')

In [ ]:
def add_break_date(sa_metadata, break_date):
    for i, row in sa_metadata.iterrows():
        if row['start_dt'] < break_date:
            if row['end_dt'] > break_date:
                sa_metadata.loc[i, 'end_dt'] = break_date - pd.Timedelta(seconds=1)
                new_row = row.copy()
                new_row['start_dt'] = break_date
                sa_metadata = pd.concat([sa_metadata, pd.DataFrame(new_row).T], ignore_index=True)
    return sa_metadata

In [ ]:
# Affix end date for current batch of sensors
arbitrary_end_date = dt(2099, 12, 31, 23, 59, 59)
last_rows_idx = sa_metadata.drop_duplicates(subset=['sensor_num'], keep='last').index
sa_metadata.loc[last_rows_idx, 'end_dt'] = arbitrary_end_date

In [ ]:
# Each CPU can only be in one sensor at a time
sa_metadata = sa_metadata.sort_values(by=['start_dt', 'sensor_num']).reset_index(drop=True)
for cpu_id in sa_metadata['cpu_serial'].unique():
    this_cpu_instances = sa_metadata[sa_metadata['cpu_serial'] == cpu_id]
    for j, this_cpu_instance in this_cpu_instances.iterrows():
        all_start_dates = this_cpu_instances['start_dt'].values
        overlap = (this_cpu_instance['start_dt'] > all_start_dates) & (this_cpu_instance['start_dt'] < this_cpu_instance['end_dt'])
        if np.any(overlap):
            for row_idx_to_check in reversed(this_cpu_instances.index):
                start_dt_to_check = sa_metadata.iloc[row_idx_to_check]['start_dt']
                rows_idx_to_modify = this_cpu_instances.iloc[this_cpu_instances.index != row_idx_to_check]['end_dt'] > start_dt_to_check
                rows_idx_to_modify = rows_idx_to_modify[rows_idx_to_modify].index
                rows_idx_to_modify = rows_idx_to_modify[rows_idx_to_modify < row_idx_to_check]
                sa_metadata.loc[rows_idx_to_modify, 'end_dt'] = start_dt_to_check - pd.Timedelta(seconds=1)

In [ ]:
# Add break for new analog boards on 2024-12-01
new_analog_boards = dt(2024, 12, 1, 0, 0, 0)
sa_metadata = add_break_date(sa_metadata, new_analog_boards)
# Add break for initial mass calibration on 2025-09-29
mass_calibration_date = dt(2025, 9, 29, 0, 0, 0)
sa_metadata = add_break_date(sa_metadata, mass_calibration_date)

In [ ]:
geoscale = np.full(len(sa_metadata), np.nan)

staticscale = np.full(len(sa_metadata), np.nan)
staticoffset = np.full(len(sa_metadata), np.nan)

massoffset = np.full(len(sa_metadata), np.nan)

channel_a_resistor = np.where(sa_metadata['start_dt'] < new_analog_boards, 10e6, 10e6)
channel_a_capacitor = np.where(sa_metadata['start_dt'] < new_analog_boards, 10e-9, 10e-9)
channel_a_gain = np.where(sa_metadata['start_dt'] < new_analog_boards, 1, 1)
channel_b_resistor = np.where(sa_metadata['start_dt'] < new_analog_boards, 1e6, 100e6)
channel_b_capacitor = np.where(sa_metadata['start_dt'] < new_analog_boards, 10e-9, 1e-9)
channel_b_gain = np.where(sa_metadata['start_dt'] < new_analog_boards, 1, 10)
channel_c_resistor = np.where(sa_metadata['start_dt'] < new_analog_boards, 10e6, 10e6)
channel_c_capacitor = np.where(sa_metadata['start_dt'] < new_analog_boards, 0.1e-6, 0.1e-6)
channel_c_gain = np.where(sa_metadata['start_dt'] < new_analog_boards, 1, 1)

In [ ]:
sa_metadata['geo_cal_scale'] = geoscale
sa_metadata['mass_cal_offset'] = massoffset
sa_metadata['channel_a_resistor_ohms'] = channel_a_resistor
sa_metadata['channel_a_capacitor_farads'] = channel_a_capacitor
sa_metadata['channel_a_RC_const_seconds'] = channel_a_resistor * channel_a_capacitor
sa_metadata['channel_a_gain'] = channel_a_gain
sa_metadata['static_cal_offset_channel_a'] = staticoffset
sa_metadata['static_cal_scale_channel_a'] = staticscale
sa_metadata['channel_b_resistor_ohms'] = channel_b_resistor
sa_metadata['channel_b_capacitor_farads'] = channel_b_capacitor
sa_metadata['channel_b_RC_const_seconds'] = channel_b_resistor * channel_b_capacitor
sa_metadata['channel_b_gain'] = channel_b_gain
sa_metadata['static_cal_offset_channel_b'] = staticoffset
sa_metadata['static_cal_scale_channel_b'] = staticscale
sa_metadata['channel_c_resistor_ohms'] = channel_c_resistor
sa_metadata['channel_c_capacitor_farads'] = channel_c_capacitor
sa_metadata['channel_c_RC_const_seconds'] = channel_c_resistor * channel_c_capacitor
sa_metadata['channel_c_gain'] = channel_c_gain
sa_metadata['static_cal_offset_channel_c'] = staticoffset
sa_metadata['static_cal_scale_channel_c'] = staticscale

In [ ]:
# Add identified date ranges to plot as bars
for i, conf in sa_metadata.iterrows():
    ax.plot([conf['start_dt'], conf['end_dt']], [conf['sensor_num']+0.1, conf['sensor_num']+0.1], color=colors[np.nonzero(cpus.index == conf['cpu_serial'])[0][0]], alpha=0.5)
    ax.plot([conf['start_dt'], conf['end_dt']], [conf['sensor_num']-0.1, conf['sensor_num']-0.1], color=colors[np.nonzero(cpus.index == conf['cpu_serial'])[0][0]], alpha=0.5)
ax.set_xlim(dt(2023, 1, 1), dt(2027, 1, 1))
fig

In [ ]:
current_config = sa_metadata.drop_duplicates(subset=['sensor_num'], keep='last').sort_values(by='sensor_num')
current_config

geocal is gain only (unitless)

static cal units:

slope is nC/V
offset V

mass test:
digital units/bits


ADC reading (bits) + mass test offset (bits) -> scale to dynmic range of ADC (V) -> apply geocal and lab cal and other buffoonery (deltaV/m)

In [ ]:
cols_to_format = [col for col in sa_metadata.columns if 'dt' not in col]
sa_metadata[cols_to_format] = sa_metadata[cols_to_format].map(lambda x: f'{x:.1e}' if isinstance(x, (int, float)) and x >= 1e6 else x)

In [ ]:
sa_metadata.to_csv('hardware.csv', index=False)